# Imports

In [9]:
from Strategies.Autotrader.TP_api import TP_api
#from Database.TPData import TPData
from datetime import datetime
import Strategies.Autotrader.enumerate as ENUM
import pandas as pd

# Setup

In [10]:
ENV = 'prod'
cls = TP_api(ENV)
cls.url, cls.token

('https://etc-api-production.visotech.com/api',
 '592ba965-64b3-4457-a83f-955b73a8b38a')

In [11]:
da_cost=0.1
wa_cost=0.045
ma_cost=0.045#+0.005
qa_cost=0.045#+0.005
cal_cost=0.045#+0.005

In [12]:
da_cent_to_eur=0.24
wa_cent_to_eur=7.2/4.0
ma_cent_to_eur=7.2
qa_cent_to_eur=21.6
cal_cent_to_eur=86.4

# Calculations

In [13]:
def calculate_arb_strategy(algo_id):
    
    if 'da' in algo_id:
        cost=da_cost
        revenue=da_cent_to_eur
    if 'weeks' in algo_id:
        cost=wa_cost
        revenue=wa_cent_to_eur
    elif 'ma' in algo_id or 'months' in algo_id or 'dec' in algo_id:
        cost=ma_cost
        revenue=ma_cent_to_eur
    elif 'qa' in algo_id:
        cost=qa_cost
        revenue=qa_cent_to_eur
    elif 'cal' in algo_id:
        cost=cal_cost
        revenue=cal_cent_to_eur
                       
    steering_list = ['stats']
    cls.steering(algo_id=algo_id,
                 steering_list=steering_list)
    
    data = cls.get_monitoring_ts(algo_id)
    aux_list = [x for x in data.json()[0]['values'] if x != None]
    #print(f'{algo_id} {aux_list}')

    res=aux_list[-1]   
    
    traded_pnl=res['pnl_dict']['pnl']
    traded_quantity=res['trade_spread_dict']['buy']['quantity']
    
    profit_eur=(traded_pnl-traded_quantity*cost)*revenue*100
    
    return algo_id, traded_quantity, profit_eur

In [14]:
active_strategies=cls.active_strategies()

In [15]:
active_strategies

['arbitrage-de-qa',
 'arbitrage-de-qa-1',
 'arbitrage-de-qa-2',
 'arbitrage-de-cal',
 'arbitrage-de-cal-1',
 'arbitrage-de-months',
 'spbmark_m_2',
 'spbmark_m_1',
 'arbitrage-de-dec',
 'mm_de_w12_v_a1',
 'mm_de_m1q1_v_a1',
 'arbitrage-de-weeks_1',
 'spbmark_m_3']

In [16]:
result=[]

for i in active_strategies:
    if 'arb' in i:
        result.append(calculate_arb_strategy(i))

{'values_from': '2024-10-16T00:00:00.000Z', 'values_until': '2024-11-15T00:00:00.000Z', 'trading_portfolio': 'arbitrage-de-qa', 'timeseries_name': 'stats'}
{'values_from': '2024-10-16T00:00:00.000Z', 'values_until': '2024-11-15T00:00:00.000Z', 'trading_portfolio': 'arbitrage-de-qa-1', 'timeseries_name': 'stats'}


IndexError: list index out of range

In [ ]:
df=pd.DataFrame(result, columns=['algo_id', 'traded_quantity', 'profit_eur'])

# Calculate column-wise totals
column_totals = df.sum().to_frame().transpose()
column_totals.index = ['Column Totals']

# Concatenate the original DataFrame with the column totals
result_df = pd.concat([df, column_totals])

print("DataFrame with Original Data and Column Totals:")
result_df